In [1]:
"""03-01 실습 정답: 유압 열화와 가열로 존 편차를 비교합니다."""

from pathlib import Path

import pandas as pd



hyd = pd.read_csv("..//3일차_실습//03-01_유압·열설비_신호_유압운전.csv", parse_dates=["date"])
furnace = pd.read_csv("..//3일차_실습//03-01_유압·열설비_신호_가열로온도.csv", parse_dates=["date"])

hyd_cols = [
    "HYD01_PRESS_PUMP", "HYD01_FLOW", "HYD01_OILTEMP",
    "HYD01_LEVEL", "HYD01_PUMP_CURRENT",
]
baseline, recent = hyd.iloc[:30], hyd.iloc[-10:]
print("=== 첫 30일 vs 마지막 10일 평균 ===")
print(pd.DataFrame({"첫30일": baseline[hyd_cols].mean(), "마지막10일": recent[hyd_cols].mean()}).round(2))

hyd["filter_dp"] = hyd["HYD01_PRESS_FILT_IN"] - hyd["HYD01_PRESS_FILT_OUT"]
print("\n=== 30일 구간별 필터 차압 ===")
for start in (0, 30, 60):
    part = hyd.iloc[start : start + 30]
    print(
        f"{start + 1}~{start + len(part)}일: "
        f"{part['filter_dp'].iloc[0]:.1f} -> {part['filter_dp'].iloc[-1]:.1f} bar "
        f"(증가 {part['filter_dp'].iloc[-1] - part['filter_dp'].iloc[0]:.1f} bar)"
    )

for zone in ("Z1", "Z2", "Z3"):
    furnace[f"{zone}_diff"] = furnace[f"FUR01_{zone}_TEMP_R"] - furnace[f"FUR01_{zone}_TEMP_L"]
print("\n=== 존별 좌우 편차 ===")
print(furnace[[f"{z}_diff" for z in ("Z1", "Z2", "Z3")]].iloc[[0, 19, 39, 59]].set_axis(
    ["1일", "20일", "40일", "60일"]
).to_string())
print("\n=== Z2 좌우 온도 ===")
print(furnace[["FUR01_Z2_TEMP_L", "FUR01_Z2_TEMP_R"]].iloc[[0, 19, 39, 59]].set_axis(
    ["1일", "20일", "40일", "60일"]
).to_string())
print("\n=== 속도 변경 전후 ===")
print(furnace[["FUR01_MAT_TEMP", "FUR01_LINE_SPEED"]].iloc[[0, 19, 43, 44, 59]].set_axis(
    ["1일", "20일", "44일", "45일", "60일"]
).to_string())
print(
    "\n해석 예시: 유압 압력·유량은 감소하고 유온·전류는 증가하지만 유면은 일정해 "
    "외부 누유만으로 설명하기 어렵습니다. 차압 증가폭도 30일 구간마다 커져 "
    "필터 막힘·계통 오염을 확인합니다."
)
print(
    "가열로 Z2의 편차는 좌측 온도 하락으로 확대됩니다. 45일에는 속도가 12에서 14 m/min으로 "
    "올라가고 소재 온도가 낮아져 체류 시간 감소가 유력한 조건 설명입니다."
)


=== 첫 30일 vs 마지막 10일 평균 ===
                      첫30일  마지막10일
HYD01_PRESS_PUMP    152.30  149.50
HYD01_FLOW          117.95  116.95
HYD01_OILTEMP        42.35   48.35
HYD01_LEVEL          88.00   88.00
HYD01_PUMP_CURRENT   31.45   32.45

=== 30일 구간별 필터 차압 ===
1~30일: 2.5 -> 5.0 bar (증가 2.5 bar)
31~60일: 2.5 -> 6.0 bar (증가 3.5 bar)
61~90일: 2.5 -> 7.0 bar (증가 4.5 bar)

=== 존별 좌우 편차 ===
     Z1_diff  Z2_diff  Z3_diff
1일       5.0      3.0      3.0
20일      6.0      4.0      4.0
40일      6.0     14.0      4.0
60일      6.0     24.0      4.0

=== Z2 좌우 온도 ===
     FUR01_Z2_TEMP_L  FUR01_Z2_TEMP_R
1일            1225.0           1228.0
20일           1224.0           1228.0
40일           1214.0           1228.0
60일           1204.0           1228.0

=== 속도 변경 전후 ===
     FUR01_MAT_TEMP  FUR01_LINE_SPEED
1일           1182.0              12.0
20일          1181.0              12.0
44일          1180.0              12.0
45일          1172.0              14.0
60일          1169.0              14.0

해석 예